# tabular_classification-iris-mlp-pytorch

Multi-class iris flower classification using a feed-forward MLP, told through `nnx`.

# 1. Overview

## 1.1 Task & motivation

Iris flower species recognition is a 75-year-old benchmark — three species (*setosa*, *versicolor*, *virginica*), four morphometric features (sepal length / width, petal length / width), 150 labeled samples evenly distributed across the classes. It's the canonical "small clean tabular classification problem" — small enough that a CPU re-run completes in seconds, clean enough that the *modeling* choices show up clearly without dataset noise drowning them out.

## 1.2 Dataset choice

The source lesson (Virginia Tech CS5644 `MLP-updated.ipynb`) used `sklearn.MLPClassifier` with 5-fold cross-validation and MinMax scaling. We keep iris and the MinMax scaler. We replace sklearn's blackbox MLP with `nnx.NNModel` so the per-epoch loss curves, validation tracking, and confusion matrices become first-class outputs of the notebook. We trade 5-fold CV for a 70 / 15 / 15 train/val/test split — at 150 samples, both regimes give comparable variance estimates, and the held-out-test split lets the candidate-comparison verdict in §6 sit on a single confusion matrix per candidate.

## 1.3 Model family

A feed-forward neural network with `nnx.FeedFwdNN` (4 input units → optional hidden layers → 3 output units, cross-entropy loss, Adam optimizer). We sweep three candidate topologies in §4 to test the *a priori* hypothesis that iris's separability is so good a linear classifier already nearly saturates it — and to quantify exactly how much (if anything) a hidden layer adds.

## 1.4 Libraries used

| Library | Used for |
|---|---|
| `nnx` | Training loop (`NNModel.train`), tabular data plumbing (`NNTabularDataset`), visualization (`VisUtils.confusion_matrix`, `VisUtils.multi_line_plot`), reproducibility (`set_seed`) |
| `torch` | Tensor backbone |
| `pandas`, `numpy`, `scikit-learn` | Iris loader, MinMax scaling, metrics, train/val/test split |
| `seaborn`, `matplotlib` | Pairplot + per-class diagnostic plots |

## 1.5 Headline result preview

The linear baseline (`hidden_dims=[]`) gets the reader to ~95 %+ test accuracy on iris. The §6 verdict will quantify whether the one-hidden-layer (`[8]`) and two-hidden-layer (`[16, 8]`) candidates close the remaining gap or just add capacity that doesn't pay for itself.


In [ ]:
SMOKE_TEST = 0
SMOKE_TEST_EPOCHS = 5


In [ ]:
# Parameters
SMOKE_TEST = 1


# 2. Environment & Setup

## 2.1 Imports

Imports are split between (a) the data + sklearn stack (iris loader, MinMax scaler, metrics, train/val/test split) and (b) the `nnx` flat re-exports introduced in the 21-commit hop (`NNModel`, `NNParams`, `NNModelParams`, `NNTrainParams`, `NNOptimParams`, `NNTabularDataset`, `Devices`, `Losses`, `Nets`, `Optims`, `set_seed`, `VisUtils`).

## 2.2 Reproducibility

`nnx.set_seed(0)` pins Python `random`, NumPy, PyTorch CPU (and CUDA if present), and cuDNN deterministic flags. We also pin `np.random.default_rng(0)` for the train/val/test split.

## 2.3 Configuration / hyperparameters

`SMOKE_TEST` papermill parameter cell controls fast-mode epoch count (`SMOKE_TEST_EPOCHS=5` when set; full `n_epochs=300` otherwise). All three §4 candidates train under the same `NNTrainParams` so the only varying axis is `hidden_dims`.


In [ ]:
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
import torch
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score, classification_report, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

import nnx
from nnx import (
    Devices,
    Losses,
    NNModel,
    NNModelParams,
    NNOptimParams,
    NNParams,
    NNRun,
    NNTabularDataset,
    NNTrainParams,
    Nets,
    Optims,
    VisUtils,
    set_seed,
)

print(f"python   = {sys.version.split()[0]}")
print(f"torch    = {torch.__version__}")
print(f"sklearn  = {sklearn.__version__}")
print(f"pandas   = {pd.__version__}")
print(f"numpy    = {np.__version__}")
print(f"nnx      = {nnx.__version__}")


In [ ]:
SEED = 0
set_seed(SEED)

n_epochs = SMOKE_TEST_EPOCHS if SMOKE_TEST else 300
print(f"SMOKE_TEST = {SMOKE_TEST}, n_epochs = {n_epochs}")
